# NHL Data Collection

This notebook collects play-by-play data from the NHL API for the 2023 regular season.

## Overview
- Generates game IDs for all regular season games
- Fetches play-by-play data from NHL API
- Saves raw data to parquet file for later processing


## Imports


In [4]:
import pandas as pd
import os
from tqdm.notebook import tqdm
import sys

# Add src to path for imports
sys.path.append('../../')
from src.data.collectors.nhl_api import get_game_data, generate_game_ids


## Constants and Configuration


In [5]:
# Configuration
SEASON = "2023"
GAME_TYPE = "02"  # 01 = Preseason, 02 = Regular Season, 03 = Playoffs
NUM_GAMES = 1312
DATA_FILE_NAME = "../../data/raw/nhl_raw_plays.parquet"  # Save to data/raw directory

# Generate all game IDs
game_ids = generate_game_ids(season=SEASON, game_type=GAME_TYPE, num_games=NUM_GAMES)

print(f"Generated {len(game_ids)} game IDs.")
print(f"First ID: {game_ids[0]}, Last ID: {game_ids[-1]}")


Generated 1312 game IDs.
First ID: 2023020001, Last ID: 2023021312


## Data Collection


In [6]:
all_plays_data = []

# Check if data file already exists
if not os.path.exists(DATA_FILE_NAME):
    print(f"Data file {DATA_FILE_NAME} does not exist. Starting data collection...")
    
    # Create data/raw directory if it doesn't exist
    os.makedirs(os.path.dirname(DATA_FILE_NAME), exist_ok=True)
    
    # Loop through all game IDs with progress bar
    for game_id in tqdm(game_ids, desc="Fetching data"):
        plays = get_game_data(game_id)
        
        # Add plays to all_plays_data
        if plays:
            all_plays_data.extend(plays)
    
    print(f"\nTotal plays collected: {len(all_plays_data)}")
    
    # Convert to DataFrame
    df = pd.DataFrame(all_plays_data)
    
    # Save to Parquet file
    df.to_parquet(DATA_FILE_NAME, index=False)
    print(f"Data saved to {DATA_FILE_NAME}")
else:
    print(f"Loading data from {DATA_FILE_NAME}...")
    df = pd.read_parquet(DATA_FILE_NAME)
    print(f"Loaded {len(df)} plays from {DATA_FILE_NAME}")

# Display first few rows of the DataFrame
print(f"\nDataFrame shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()


Loading data from ../../data/raw/nhl_raw_plays.parquet...
Loaded 413676 plays from ../../data/raw/nhl_raw_plays.parquet

DataFrame shape: (413676, 11)

Columns: ['eventId', 'periodDescriptor', 'timeInPeriod', 'timeRemaining', 'situationCode', 'homeTeamDefendingSide', 'typeCode', 'typeDescKey', 'sortOrder', 'details', 'pptReplayUrl']


,eventId,periodDescriptor,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,details,pptReplayUrl
0,102,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:00,20:00,1551,left,520,period-start,8,None,None
1,101,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:00,20:00,1551,left,502,faceoff,9,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
2,8,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:35,19:25,1551,left,516,stoppage,15,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
3,103,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:35,19:25,1551,left,502,faceoff,17,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
4,9,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:48,19:12,1551,left,503,hit,20,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
